In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('datasets/1_clean_data_train.csv')

In [3]:
df.columns

Index(['LengthOfURL', 'DomainLengthOfURL', 'IsDomainIP', 'TLDLength',
       'URLLetterRatio', 'DigitCntInURL', 'EqualCharCntInURL',
       'QuesMarkCntInURL', 'OtherSpclCharCntInURL', 'HavingPath', 'PathLength',
       'HavingAnchor', 'HasSSL', 'IsUnreachable', 'LineOfCode', 'HasTitle',
       'HasFavicon', 'HasRobotsBlocked', 'IsResponsive', 'IsURLRedirects',
       'IsSelfRedirects', 'HasDescription', 'IsFormSubmitExternal',
       'HasSocialMediaPage', 'HasSubmitButton', 'HasHiddenFields',
       'HasPasswordFields', 'HasBankingKey', 'HasPaymentKey',
       'HasCopyrightInfoKey', 'CntImages', 'CntFilesCSS', 'CntFilesJS',
       'CntSelfHRef', 'CntEmptyRef', 'CntExternalRef', 'CntIFrame',
       'UniqueFeatureCnt', 'ShannonEntropy', 'FractalDimension',
       'KolmogorovComplexity', 'HexPatternCnt', 'Base64PatternCnt',
       'LikelinessIndex', 'Label'],
      dtype='object')

----------------URL VE SAYFA İÇERİĞİ ÖZELLİKLERİYLE -----------------------

In [4]:
df_all_feature = df

In [5]:
df_url = df

In [6]:
df_all_feature.duplicated().sum()

np.int64(29083)

In [7]:
df_all_feature = df_all_feature.drop_duplicates()

In [8]:
x = df_all_feature.drop("Label", axis=1)
y = df_all_feature["Label"].values.ravel()

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.naive_bayes import BernoulliNB

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

def algo_test(x, y, save_models=False):

    models = {
        "BernoulliNB": BernoulliNB(),
        "LogisticRegression": LogisticRegression(max_iter=1000),
        "DecisionTreeClassifier": DecisionTreeClassifier(),
        "RandomForestClassifier": RandomForestClassifier(),
        "GradientBoostingClassifier": GradientBoostingClassifier(),
        "KNeighborsClassifier": KNeighborsClassifier(),
        "AdaBoostClassifier": AdaBoostClassifier()
    }

    # split
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=0.3, random_state=42, stratify=y
    )

    results = []

    print("Veriler hazır, modeller deneniyor...")

    for name, model in models.items():

        print(f"{name} modeli eğitiliyor...")

        # Scale isteyen modeller
        if name in ["LogisticRegression", "KNeighborsClassifier"]:
            pipeline = Pipeline([
                ("scaler", StandardScaler()),
                ("model", model)
            ])
            pipeline.fit(x_train, y_train)
            y_pred = pipeline.predict(x_test)
            trained_model = pipeline

        else:
            model.fit(x_train, y_train)
            y_pred = model.predict(x_test)
            trained_model = model

        results.append({
            "Model": name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "F1": f1_score(y_test, y_pred),
            "TrainedModel": trained_model
        })

        print(confusion_matrix(y_test, y_pred))

    df_results = pd.DataFrame(results).sort_values("F1", ascending=False)

    print("\nEn başarılı model:", df_results.iloc[0]["Model"])

    if save_models:
        import os, joblib
        os.makedirs("models", exist_ok=True)

        for row in results:
            joblib.dump(row["TrainedModel"], f"models/{row['Model']}.pkl")

        print("Modeller kaydedildi.")

    return df_results.drop("TrainedModel", axis=1)

In [10]:
algo_test(x,y,save_models =False)

Veriler hazır, modeller deneniyor...
BernoulliNB modeli eğitiliyor...
[[45911  2290]
 [  204 43893]]
LogisticRegression modeli eğitiliyor...
[[48161    40]
 [  197 43900]]
DecisionTreeClassifier modeli eğitiliyor...
[[48148    53]
 [   73 44024]]
RandomForestClassifier modeli eğitiliyor...
[[48178    23]
 [   60 44037]]
GradientBoostingClassifier modeli eğitiliyor...
[[48157    44]
 [   66 44031]]
KNeighborsClassifier modeli eğitiliyor...
[[48125    76]
 [  211 43886]]
AdaBoostClassifier modeli eğitiliyor...
[[48095   106]
 [  229 43868]]

En başarılı model: RandomForestClassifier


,Model,Accuracy,Precision,Recall,F1
3,RandomForestClassifier,0.999101,0.999478,0.998639,0.999058
4,GradientBoostingClassifier,0.998808,0.999002,0.998503,0.998752
2,DecisionTreeClassifier,0.998635,0.998798,0.998345,0.998571
1,LogisticRegression,0.997432,0.999090,0.995533,0.997308
5,KNeighborsClassifier,0.996891,0.998271,0.995215,0.996741
6,AdaBoostClassifier,0.996370,0.997589,0.994807,0.996196
0,BernoulliNB,0.972979,0.950415,0.995374,0.972375


---------------- URL TABANLI ÖZELLİKLERLE EĞİTİM ALMA -----------------------

In [11]:
df_url = df_url[
    [
        # Uzunluk & Domain
        "LengthOfURL",
        "DomainLengthOfURL",
        "TLDLength",
        "IsDomainIP",

        # Karakter & Yapı
        "DigitCntInURL",
        "EqualCharCntInURL",
        "QuesMarkCntInURL",
        "OtherSpclCharCntInURL",
        "URLLetterRatio",
        "HavingPath",
        "HavingAnchor",
        "PathLength",

        # Matematiksel / Metin Karmaşıklığı
        "ShannonEntropy",
        "FractalDimension",
        "KolmogorovComplexity",
        "HexPatternCnt",
        "Base64PatternCnt",

        # Hedef
        "Label",
    ]
]

In [12]:
df_url.duplicated().sum()

np.int64(195792)

In [13]:
df_url = df_url.drop_duplicates()

In [14]:
df_url.head(10)

,LengthOfURL,DomainLengthOfURL,TLDLength,IsDomainIP,DigitCntInURL,EqualCharCntInURL,QuesMarkCntInURL,OtherSpclCharCntInURL,URLLetterRatio,HavingPath,HavingAnchor,PathLength,ShannonEntropy,FractalDimension,KolmogorovComplexity,HexPatternCnt,Base64PatternCnt,Label
0,84,76,3,0,17,0,0,4,0.655,0,0,0,4.739567,1.00,1.000000,1,3,1
1,40,20,8,1,17,0,0,6,0.250,2,0,1,3.808271,1.00,1.222222,5,3,1
2,28,19,3,0,2,0,0,3,0.536,1,0,1,4.056021,0.75,1.307692,0,2,1
3,31,23,3,0,0,0,0,3,0.677,1,0,1,3.827833,0.75,1.275862,0,2,1
4,23,15,2,0,0,0,0,2,0.391,0,0,0,3.346439,1.00,1.400000,0,2,0
5,34,25,3,0,0,0,0,4,0.647,1,0,1,3.965018,0.75,1.250000,0,3,1
6,38,30,3,0,0,0,0,4,0.579,0,0,0,3.917626,1.00,1.235294,0,3,1
7,28,20,2,0,0,0,0,1,0.536,0,0,0,3.719295,1.00,1.307692,0,2,0
8,22,7,2,0,1,0,0,2,0.500,1,0,7,4.070656,1.00,1.380952,1,2,1
9,26,18,3,0,0,0,0,1,0.500,0,0,0,3.636842,1.00,1.333333,0,2,0


In [15]:
x = df_url.drop("Label", axis=1)
y = df_url["Label"].values.ravel()

In [16]:
x.head(10)

,LengthOfURL,DomainLengthOfURL,TLDLength,IsDomainIP,DigitCntInURL,EqualCharCntInURL,QuesMarkCntInURL,OtherSpclCharCntInURL,URLLetterRatio,HavingPath,HavingAnchor,PathLength,ShannonEntropy,FractalDimension,KolmogorovComplexity,HexPatternCnt,Base64PatternCnt
0,84,76,3,0,17,0,0,4,0.655,0,0,0,4.739567,1.00,1.000000,1,3
1,40,20,8,1,17,0,0,6,0.250,2,0,1,3.808271,1.00,1.222222,5,3
2,28,19,3,0,2,0,0,3,0.536,1,0,1,4.056021,0.75,1.307692,0,2
3,31,23,3,0,0,0,0,3,0.677,1,0,1,3.827833,0.75,1.275862,0,2
4,23,15,2,0,0,0,0,2,0.391,0,0,0,3.346439,1.00,1.400000,0,2
5,34,25,3,0,0,0,0,4,0.647,1,0,1,3.965018,0.75,1.250000,0,3
6,38,30,3,0,0,0,0,4,0.579,0,0,0,3.917626,1.00,1.235294,0,3
7,28,20,2,0,0,0,0,1,0.536,0,0,0,3.719295,1.00,1.307692,0,2
8,22,7,2,0,1,0,0,2,0.500,1,0,7,4.070656,1.00,1.380952,1,2
9,26,18,3,0,0,0,0,1,0.500,0,0,0,3.636842,1.00,1.333333,0,2


In [20]:
algo_test(x,y,save_models=True)

Veriler hazır, modeller deneniyor...
BernoulliNB modeli eğitiliyor...
[[ 6956   204]
 [ 6422 28703]]
LogisticRegression modeli eğitiliyor...
[[ 6641   519]
 [ 1201 33924]]
DecisionTreeClassifier modeli eğitiliyor...
[[ 6917   243]
 [  292 34833]]
RandomForestClassifier modeli eğitiliyor...
[[ 7000   160]
 [  228 34897]]
GradientBoostingClassifier modeli eğitiliyor...
[[ 6936   224]
 [  836 34289]]
KNeighborsClassifier modeli eğitiliyor...
[[ 6864   296]
 [  911 34214]]
AdaBoostClassifier modeli eğitiliyor...
[[ 6540   620]
 [ 1774 33351]]

En başarılı model: RandomForestClassifier
Modeller kaydedildi.


,Model,Accuracy,Precision,Recall,F1
3,RandomForestClassifier,0.990824,0.995436,0.993509,0.994472
2,DecisionTreeClassifier,0.987348,0.993072,0.991687,0.992379
4,GradientBoostingClassifier,0.974932,0.993510,0.976199,0.984778
5,KNeighborsClassifier,0.971456,0.991423,0.974064,0.982667
1,LogisticRegression,0.959324,0.984932,0.965808,0.975276
6,AdaBoostClassifier,0.943384,0.981749,0.949495,0.965353
0,BernoulliNB,0.843301,0.992943,0.817167,0.896520


In [22]:
# modelin doğruluğu hangi özelliğe daha çok bağlı olduğunu görmek için . Modele etkisini 
gb = GradientBoostingClassifier()
gb.fit(df_url.drop("Label", axis=1), y)

importances = pd.Series(
    gb.feature_importances_,
    index=df_url.drop("Label", axis=1).columns
)

print(importances.sort_values(ascending=False).head(10))

PathLength               0.595776
DigitCntInURL            0.084727
OtherSpclCharCntInURL    0.079621
URLLetterRatio           0.064616
LengthOfURL              0.045040
TLDLength                0.032802
HavingPath               0.032376
DomainLengthOfURL        0.025536
KolmogorovComplexity     0.020070
HavingAnchor             0.013702
dtype: float64


In [23]:
# PathLength çıkarıldıktan sonra modelin doğruluk oranındaki değişiklik
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import GradientBoostingClassifier

x_full = df_url.drop("Label", axis=1)

x_without_path = x_full.drop("PathLength", axis=1)

model = GradientBoostingClassifier(random_state=42)

score_full = cross_val_score(model, x_full, y, cv=5, scoring="f1").mean()
score_without = cross_val_score(model, x_without_path, y, cv=5, scoring="f1").mean()

print("Full:", score_full)
print("Without PathLength:", score_without)
print("Fark:", score_full - score_without)

Full: 0.9846640657948038
Without PathLength: 0.9834468460063615
Fark: 0.0012172197884423586


In [24]:
# PathLength ÖNEMLİ BİR ÖZELLİK ANCAK ÇIKARILDIĞINDA MODELİN DOĞRULUĞU ÇOK DÜŞMEDİĞİ İÇİN DİĞER ÖZELLİKLERDE SONUÇTA ETKİLİ

In [25]:
df_url.duplicated().sum()

np.int64(0)

In [29]:
df_url.dtypes

LengthOfURL                int64
DomainLengthOfURL          int64
TLDLength                  int64
IsDomainIP                 int64
DigitCntInURL              int64
EqualCharCntInURL          int64
QuesMarkCntInURL           int64
OtherSpclCharCntInURL      int64
URLLetterRatio           float64
HavingPath                 int64
HavingAnchor               int64
PathLength                 int64
ShannonEntropy           float64
FractalDimension         float64
KolmogorovComplexity     float64
HexPatternCnt              int64
Base64PatternCnt           int64
Label                      int64
dtype: object

In [30]:
print("TRAIN STATS")
print(df_url.describe())

TRAIN STATS
         LengthOfURL  DomainLengthOfURL      TLDLength     IsDomainIP  \
count  140950.000000      140950.000000  140950.000000  140950.000000   
mean       54.532189          24.638170       2.952969       0.086676   
std        36.447957          13.382472       1.052817       0.281361   
min        14.000000           4.000000       1.000000       0.000000   
25%        31.000000          15.000000       2.000000       0.000000   
50%        40.000000          22.000000       3.000000       0.000000   
75%        66.000000          29.000000       3.000000       0.000000   
max       200.000000         123.000000      13.000000       1.000000   

       DigitCntInURL  EqualCharCntInURL  QuesMarkCntInURL  \
count  140950.000000      140950.000000     140950.000000   
mean        5.912990           0.215793          0.094871   
std         8.799882           0.796022          0.302027   
min         0.000000           0.000000          0.000000   
25%         0.000000     